# 05 — Tree-Based Valuation Models

## Plate Value Intelligence

This notebook evaluates non-linear tree-based models for predicting DVLA auction hammer prices.

### Objectives

- Preserve the approved event-based validation design
- Reuse the governed, leakage-safe feature set
- Train non-linear valuation models
- Compare performance against Ridge regression and rule-based baselines
- Analyse premium-asset prediction weaknesses
- Select the strongest candidate for final holdout testing

### Modelling principle

Model selection will use the B277 validation event only. The B278 test event remains untouched until the final model and configuration have been selected.

In [4]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import (
    HistGradientBoostingRegressor,
    RandomForestRegressor,
)
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    median_absolute_error,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

In [5]:
PROJECT_ROOT = Path.cwd()

FEATURE_FILE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "plate_features_2025.csv"
)

REPORTS_DIR = PROJECT_ROOT / "reports"

print(f"Feature file exists: {FEATURE_FILE.exists()}")

Feature file exists: True


In [8]:
df = pd.read_csv(FEATURE_FILE)

validation_group_map = {
    "B270": "train",
    "B271": "train",
    "B272": "train",
    "B273": "train",
    "B274": "train",
    "B275": "train",
    "B276": "train",
    "B277": "validation",
    "B278": "test",
}

df["model_split"] = df["event_code"].map(validation_group_map)

missing_split_events = sorted(
    df.loc[df["model_split"].isna(), "event_code"].dropna().unique()
)
if missing_split_events:
    raise ValueError(f"Missing model_split mapping for events: {missing_split_events}")

print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]}")
print(f"Model splits: {sorted(df['model_split'].unique())}")

Rows: 17,782
Columns: 76
Model splits: ['test', 'train', 'validation']


## 1. Event-Based Split Validation

The approved chronological event split is validated before any model training.

In [9]:
expected_split_map = {
    "B270": "train",
    "B271": "train",
    "B272": "train",
    "B273": "train",
    "B274": "train",
    "B275": "train",
    "B276": "train",
    "B277": "validation",
    "B278": "test",
}

actual_split_map = (
    df[["event_code", "model_split"]]
    .drop_duplicates()
    .set_index("event_code")["model_split"]
    .to_dict()
)

assert actual_split_map == expected_split_map, (
    "Event-based split does not match the approved design."
)

print("Event-based split is correct.")

Event-based split is correct.


In [10]:
train_df = df[df["model_split"].eq("train")].copy()
validation_df = df[df["model_split"].eq("validation")].copy()
test_df = df[df["model_split"].eq("test")].copy()

print(f"Train rows: {len(train_df):,}")
print(f"Validation rows: {len(validation_df):,}")
print(f"Test rows: {len(test_df):,}")

Train rows: 13,824
Validation rows: 1,977
Test rows: 1,981


## 2. Leakage-Safe Predictor Set

Only pre-sale structural features are used as predictors. Price-derived premium labels, targets, identifiers, and source-governance fields are excluded.

In [12]:
TARGET_COLUMN = "log_hammer_price"
RAW_PRICE_COLUMN = "hammer_price_gbp"

NUMERIC_FEATURES = [
    "plate_length",
    "letter_count",
    "digit_count",
    "numeric_length",
    "numeric_value",
    "unique_character_count",
    "repeated_character_count",
    "maximum_character_run",
    "segment_count",
    "first_segment_length",
    "last_segment_length",
    "block_count",
    "longest_block_length",
    "first_numeric_value",
    "last_numeric_value",
    "low_number_score",
    "is_single_digit_number",
    "is_two_digit_number",
    "is_number_below_100",
    "is_number_below_1000",
    "has_repeated_digit",
    "has_repeated_letter",
    "all_digits_same",
    "all_letters_same",
    "is_palindrome",
    "has_sequential_pattern",
    "is_short_plate_len_3_or_less",
    "is_short_plate_len_4_or_less",
    "is_round_number",
    "is_milestone_number",
]

CATEGORICAL_FEATURES = [
    "plate_pattern",
    "plate_format_group",
]

MODEL_FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

In [13]:
missing_features = [
    column
    for column in MODEL_FEATURES
    if column not in df.columns
]

assert not missing_features, (
    f"Missing model features: {missing_features}"
)

for forbidden_column in [
    "hammer_price_gbp",
    "log_hammer_price",
    "is_premium_top_10pct",
    "is_premium_top_5pct",
    "is_premium_top_1pct",
]:
    assert forbidden_column not in MODEL_FEATURES

print("Feature set passed leakage checks.")

Feature set passed leakage checks.


In [14]:
X_train = train_df[MODEL_FEATURES].copy()
X_validation = validation_df[MODEL_FEATURES].copy()
X_test = test_df[MODEL_FEATURES].copy()

y_train = train_df[TARGET_COLUMN].copy()
y_validation = validation_df[TARGET_COLUMN].copy()
y_test = test_df[TARGET_COLUMN].copy()

print(f"X_train: {X_train.shape}")
print(f"X_validation: {X_validation.shape}")
print(f"X_test: {X_test.shape}")

X_train: (13824, 32)
X_validation: (1977, 32)
X_test: (1981, 32)


In [15]:
def evaluate_price_predictions(
    actual_prices,
    predicted_prices,
    model_name,
    dataset_name,
):
    actual_prices = np.asarray(actual_prices, dtype=float)
    predicted_prices = np.asarray(predicted_prices, dtype=float)

    predicted_prices = np.clip(
        predicted_prices,
        a_min=0,
        a_max=None,
    )

    return {
        "model": model_name,
        "dataset": dataset_name,
        "mae_gbp": mean_absolute_error(
            actual_prices,
            predicted_prices,
        ),
        "median_absolute_error_gbp": median_absolute_error(
            actual_prices,
            predicted_prices,
        ),
        "rmse_gbp": np.sqrt(
            mean_squared_error(
                actual_prices,
                predicted_prices,
            )
        ),
        "log_rmse": np.sqrt(
            mean_squared_error(
                np.log1p(actual_prices),
                np.log1p(predicted_prices),
            )
        ),
    }

In [16]:
tree_numeric_preprocessor = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median"),
        ),
    ]
)

tree_categorical_preprocessor = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent"),
        ),
        (
            "one_hot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False,
            ),
        ),
    ]
)

tree_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            tree_numeric_preprocessor,
            NUMERIC_FEATURES,
        ),
        (
            "categorical",
            tree_categorical_preprocessor,
            CATEGORICAL_FEATURES,
        ),
    ],
    remainder="drop",
)

## 3. Random Forest Valuation Model

Random Forest can learn non-linear relationships and interactions without requiring those interactions to be manually specified.

In [17]:
random_forest_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            tree_preprocessor,
        ),
        (
            "model",
            RandomForestRegressor(
                n_estimators=300,
                max_depth=14,
                min_samples_leaf=5,
                max_features="sqrt",
                random_state=42,
                n_jobs=-1,
            ),
        ),
    ]
)

In [18]:
random_forest_pipeline.fit(
    X_train,
    y_train,
)

print("Random Forest trained successfully.")

Random Forest trained successfully.


In [19]:
rf_validation_predictions_log = (
    random_forest_pipeline.predict(
        X_validation
    )
)

rf_validation_predictions_gbp = np.expm1(
    rf_validation_predictions_log
)

rf_validation_predictions_gbp = np.clip(
    rf_validation_predictions_gbp,
    a_min=0,
    a_max=None,
)

In [20]:
tree_model_results = []

tree_model_results.append(
    evaluate_price_predictions(
        actual_prices=validation_df[
            RAW_PRICE_COLUMN
        ],
        predicted_prices=rf_validation_predictions_gbp,
        model_name="random_forest",
        dataset_name="validation",
    )
)

tree_model_results_df = pd.DataFrame(
    tree_model_results
)

tree_model_results_df

,model,dataset,mae_gbp,median_absolute_error_gbp,rmse_gbp,log_rmse
0,random_forest,validation,"1,230.53",638.05,"2,348.75",0.75


## 4. Histogram Gradient Boosting Model

Histogram Gradient Boosting builds trees sequentially, with each new tree focusing on errors made by the previous trees. It is evaluated as a stronger non-linear candidate for plate valuation.

In [21]:
hist_gradient_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            tree_preprocessor,
        ),
        (
            "model",
            HistGradientBoostingRegressor(
                learning_rate=0.05,
                max_iter=300,
                max_leaf_nodes=31,
                min_samples_leaf=20,
                l2_regularization=1.0,
                random_state=42,
            ),
        ),
    ]
)